# Seven-Day Multi-Model Forecast Research

This notebook is an interpretation layer over reusable code in `src/forecasting.py` and precomputed artifacts in `output/forecasting/`. It does not train models. Regenerate artifacts explicitly with `python main.py train-models --force`.

> Research output only. The source provenance is unverified, and these are not official HHS or CBP forecasts.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecasting import dependency_status  # noqa: E402

ARTIFACT_ROOT = PROJECT_ROOT / 'output' / 'forecasting'
dependency_status()

## Provenance and leakage audit

The audit must pass before any generated performance result is interpreted.

In [ ]:
provenance = json.loads((ARTIFACT_ROOT / 'audits' / 'dataset_provenance.json').read_text())
leakage = json.loads((ARTIFACT_ROOT / 'audits' / 'leakage_audit.json').read_text())
promotion = json.loads((ARTIFACT_ROOT / 'metrics' / 'promotion_decision.json').read_text())
pd.Series({
    'provenance_classification': provenance['classification'],
    'local_lineage_hash_matches': provenance['local_lineage_hash_matches'],
    'contains_personal_data': provenance['contains_personal_or_child_level_data'],
    'leakage_audit_passed': leakage['passed'],
    'champion': promotion['champion_model'],
    'promotion': promotion['recommendation'],
})

## Common model comparison

In [ ]:
metrics = json.loads((ARTIFACT_ROOT / 'metrics' / 'model_comparison_metrics.json').read_text())
comparison = pd.DataFrame([
    {
        'model': name,
        'walk_forward_mae': values['walk_forward']['mean_mae'],
        'walk_forward_sd': values['walk_forward']['std_mae'],
        'worst_fold_mae': values['walk_forward']['worst_fold_mae'],
        'holdout_mae': values['holdout']['mae'],
        'holdout_mase': values['holdout']['mase_vs_persistence'],
    }
    for name, values in metrics['models'].items()
]).sort_values('walk_forward_mae')
comparison

In [ ]:
px.bar(
    comparison,
    x='model',
    y=['walk_forward_mae', 'holdout_mae'],
    barmode='group',
    title='Development and untouched-holdout MAE',
).show()

## Holdout forecast and calibrated interval

In [ ]:
predictions = pd.read_csv(
    ARTIFACT_ROOT / 'predictions' / 'final_holdout_predictions.csv',
    parse_dates=['forecast_origin_date', 'target_date'],
)
champion = promotion['champion_model']
view = predictions.query('model_name == @champion').sort_values('target_date')
figure = go.Figure()
figure.add_scatter(x=view.target_date, y=view.upper_interval, line={'width': 0}, showlegend=False)
figure.add_scatter(x=view.target_date, y=view.lower_interval, line={'width': 0}, fill='tonexty', name='80% interval')
figure.add_scatter(x=view.target_date, y=view.actual_value, name='Actual')
figure.add_scatter(x=view.target_date, y=view.reconstructed_absolute_prediction, name=champion)
figure.update_layout(title='Untouched holdout forecast', template='plotly_white')
figure.show()

## Diagnostics

Permutation importance is predictive rather than causal and can be divided across correlated features.

In [ ]:
importance = pd.read_csv(ARTIFACT_ROOT / 'diagnostics' / 'oof_permutation_importance.csv')
regimes = pd.read_csv(ARTIFACT_ROOT / 'diagnostics' / 'error_by_regime.csv')
display(importance.head(20))
display(regimes)